<a href="https://colab.research.google.com/github/rotoncsedu/ESA-assignment/blob/main/BERT_CRF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
pip install pytorch-crf

!pip install seqeval

In [18]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertModel
from torchcrf import CRF
import spacy


# Load spaCy
nlp = spacy.load("en_core_web_sm")

# Weak Labeling

def weak_labeling(text):
    doc = nlp(text)

    tokens = [token.text for token in doc]
    labels = ["O"] * len(tokens)

    # OPINION = adjectives
    for i, token in enumerate(doc):
        if token.pos_ == "ADJ":
            labels[i] = "B-OPINION"

    # FEATURE = noun chunks
    for chunk in doc.noun_chunks:
        start = chunk.start
        end = chunk.end
        labels[start] = "B-FEATURE"
        for i in range(start + 1, end):
            labels[i] = "I-FEATURE"

    # Cleaning
    for i, token in enumerate(doc):
        if token.lower_ in ["a","an","the","have","has","is","are","and"]:
            labels[i] = "O"


    return tokens, labels

# Load Dataset

def load_data(file_path, max_samples=2000):
    texts = []

    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            item = json.loads(line)
            if "reviewText" in item:
                texts.append(item["reviewText"])

    dataset = []
    for text in texts:
        tokens, labels = weak_labeling(text)
        dataset.append((tokens, labels))

    return dataset

# Label Mapping

label_list = ["O", "B-FEATURE", "I-FEATURE", "B-OPINION"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}


# Dataset Class

class ReviewDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids()
        aligned_labels = []

        prev_word = None

        for word_id in word_ids:
            if word_id is None:
                # For special tokens ([CLS], [SEP]) or padding tokens, assign 'O' label
                aligned_labels.append(label2id['O'])
            elif word_id != prev_word:
                # Assign label for the first subword token of a word
                aligned_labels.append(label2id[labels[word_id]])
            else:
                # Assign label for subsequent subword tokens of the same word
                aligned_labels.append(label2id[labels[word_id]])
            prev_word = word_id

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(aligned_labels)
        }

# BERT + CRF Model

class BERT_CRF(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        emissions = self.classifier(sequence_output)

        if labels is not None:
            mask = attention_mask.bool()
            loss = -self.crf(emissions, labels, mask=mask, reduction='mean')
            return loss
        else:
            mask = attention_mask.bool()
            prediction = self.crf.decode(emissions, mask=mask)
            return prediction

# Load

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

data = load_data("reviews.json", max_samples=2000)

dataset = ReviewDataset(data, tokenizer)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = BERT_CRF(num_labels=len(label_list)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

# Training

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        loss = model(input_ids, attention_mask, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


# Prediction Function

def predict(text):
    model.eval()

    tokens = text.split()
    encoding = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    preds = model(input_ids, attention_mask)

    tags = [id2label[p] for p in preds[0]]

    return list(zip(tokens, tags))

#  Test

print(predict("this cover makes an old phone look and feel new. I like that I can order covers for little money and snazzy up my phone."))

['Looks', 'even', 'better', 'in', 'person', '.', 'Be', 'careful', 'to', 'not', 'drop', 'your', 'phone', 'so', 'often', 'because', 'the', 'rhinestones', 'will', 'fall', 'off', '(', 'duh', ')', '.', 'More', 'of', 'a', 'decorative', 'case', 'than', 'it', 'is', 'protective', ',', 'but', 'I', 'will', 'say', 'that', 'it', 'fits', 'perfectly', 'and', 'securely', 'on', 'my', 'phone', '.', 'Overall', ',', 'very', 'pleased', 'with', 'this', 'purchase', '.']
['O', 'O', 'B-OPINION', 'O', 'B-FEATURE', 'O', 'O', 'B-OPINION', 'O', 'O', 'O', 'B-FEATURE', 'I-FEATURE', 'O', 'O', 'O', 'O', 'I-FEATURE', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-OPINION', 'O', 'O', 'I-FEATURE', 'I-FEATURE', 'O', 'B-FEATURE', 'O', 'B-OPINION', 'O', 'O', 'B-FEATURE', 'O', 'O', 'O', 'B-FEATURE', 'O', 'O', 'O', 'O', 'O', 'B-FEATURE', 'I-FEATURE', 'O', 'O', 'O', 'O', 'B-OPINION', 'O', 'B-FEATURE', 'I-FEATURE', 'O']
['When', 'you', 'do', "n't", 'want', 'to', 'spend', 'a', 'whole', 'lot', 'of', 'cash', 'but', 'want', 'a', 'great', 'd

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1, Loss: 748.4306
Epoch 2, Loss: 320.0907
Epoch 3, Loss: 165.3989
Epoch 4, Loss: 100.7892
Epoch 5, Loss: 66.1559
[('this', 'O'), ('cover', 'B-FEATURE'), ('makes', 'I-FEATURE'), ('an', 'O'), ('old', 'O'), ('phone', 'I-FEATURE'), ('look', 'I-FEATURE'), ('and', 'I-FEATURE'), ('feel', 'O'), ('new.', 'O'), ('I', 'B-OPINION'), ('like', 'O'), ('that', 'B-FEATURE'), ('I', 'O'), ('can', 'B-FEATURE'), ('order', 'B-FEATURE'), ('covers', 'O'), ('for', 'O'), ('little', 'B-FEATURE'), ('money', 'O'), ('and', 'B-FEATURE'), ('snazzy', 'I-FEATURE'), ('up', 'O'), ('my', 'B-OPINION'), ('phone.', 'B-OPINION')]


In [19]:
pip install seqeval

In [20]:
from seqeval.metrics import classification_report, f1_score

def evaluate(model, dataloader):
    model.eval()

    true_labels = []
    pred_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            predictions = model(input_ids, attention_mask)

            for i in range(len(predictions)):
                pred_seq = predictions[i]
                true_seq = labels[i].cpu().numpy()

                temp_true = []
                temp_pred = []

                for t, p in zip(true_seq, pred_seq):
                    if t == -100:
                        continue
                    temp_true.append(id2label[t])
                    temp_pred.append(id2label[p])

                true_labels.append(temp_true)
                pred_labels.append(temp_pred)

    print(classification_report(true_labels, pred_labels))
    print("F1-score:", f1_score(true_labels, pred_labels))

In [21]:
evaluate(model, loader)

              precision    recall  f1-score   support

     FEATURE       0.97      0.98      0.97      1672
     OPINION       0.93      0.98      0.95       283

   micro avg       0.97      0.98      0.97      1955
   macro avg       0.95      0.98      0.96      1955
weighted avg       0.97      0.98      0.97      1955

F1-score: 0.9712395011453296
